# Visualization 1

## Cleaning Data

In [10]:
import pandas as pd
import altair as alt

alt.data_transformers.enable('json')

data = pd.read_csv('data/dpt2020.csv',sep=";")

In [16]:
df = data[(data["annais"] != "XXXX") & (data["dpt"] != "XX")]

In [17]:
df.head()

,sexe,preusuel,annais,dpt,nombre
0,1,_PRENOMS_RARES,1900,02,7
1,1,_PRENOMS_RARES,1900,04,9
2,1,_PRENOMS_RARES,1900,05,8
3,1,_PRENOMS_RARES,1900,06,23
4,1,_PRENOMS_RARES,1900,07,9


In [46]:
repartition_sexe = df.groupby(['preusuel', 'sexe'])['nombre'].sum().unstack(fill_value=0)

repartition_sexe['total'] = repartition_sexe.sum(axis=1)
repartition_sexe['part_minoritaire'] = repartition_sexe.drop(columns=['total']).min(axis=1) / repartition_sexe['total']

# Il y a du bruit, Marie a parfois été annoté comme garcon : règle de 10%.
vrais_epicenes = repartition_sexe[(repartition_sexe['part_minoritaire'] >= 0.10) & (repartition_sexe['total'] >= 500)].index
df_filtre = df[df['preusuel'].isin(vrais_epicenes)]
evolution_annuelle = df_filtre.groupby(['preusuel', 'sexe', 'annais'])['nombre'].sum().reset_index()

evolution_pivot = evolution_annuelle.pivot_table(
    index=['preusuel', 'annais'], 
    columns='sexe', 
    values='nombre', 
    fill_value=0
)

# Certain prénom n'ont été donné que sur très peu de de temps, corrélation peu informative : filtre de robustesse (exige au moins 10 ans d'historique)
comptage_annees = evolution_annuelle.groupby('preusuel')['annais'].nunique()
prenoms_solides = comptage_annees[comptage_annees >= 10].index
evolution_pivot_filtre = evolution_pivot.loc[prenoms_solides]

correlations = evolution_pivot_filtre.groupby('preusuel').apply(
    lambda x: x.iloc[:, 0].corr(x.iloc[:, 1], min_periods=5)
).reset_index(name='correlation_H_F')

prenoms_epicenes = df_filtre.groupby(['preusuel', 'sexe'])['nombre'].sum().reset_index()
prenoms_epicenes_pop = prenoms_epicenes.groupby(['preusuel'])['nombre'].sum().reset_index()
prenoms_epicenes_pop = prenoms_epicenes_pop.merge(correlations, on='preusuel', how='left')
prenoms_epicenes_pop = prenoms_epicenes_pop.sort_values(by='correlation_H_F', ascending=True)

In [41]:
prenoms_epicenes_pop.head()

,preusuel,nombre,correlation_H_F_x,correlation_H_F_y
0,MARIE,2256072,0.922044,0.922044
1,JEAN,1913130,0.605867,0.605867
2,_PRENOMS_RARES,1651579,0.998885,0.998885
3,PIERRE,891794,0.721591,0.721591
4,MICHEL,818025,0.038839,0.038839


In [48]:
brush = alt.selection_interval()

scatter_plot = alt.Chart(prenoms_epicenes_pop).mark_circle(size=60).encode(
    x=alt.X('nombre:Q', 
            scale=alt.Scale(type='log'), 
            title='Popularité totale (cumul H+F) - Échelle Log'),
    
    y=alt.Y('correlation_H_F:Q', 
            title='Corrélation Homme / Femme (-1 à 1)'),
    
    tooltip=[
        alt.Tooltip('preusuel:N', title='Prénom'),
        alt.Tooltip('nombre:Q', title='Popularité totale'),
        alt.Tooltip('correlation_H_F:Q', title='Corrélation H/F', format='.2f')
    ],
    
    color=alt.condition(brush, alt.value('steelblue'), alt.value('lightgray')),
    
    opacity=alt.condition(brush, alt.value(0.8), alt.value(0.2))
).add_params(
    brush
).properties(
    width=700,
    height=500,
    title='Analyse des prénoms épicènes : Popularité vs Corrélation des genres'
)
scatter_plot

alt.Chart(...)